# Format

In [1]:
import numpy as np
import os
import json
import pandas as pd

In [2]:
# DP3, 2024 data
data_df = pd.read_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP05.csv")
data_df.head()

,DP05_0001E,DP05_0001EA,DP05_0001M,DP05_0001MA,DP05_0001PE,DP05_0001PEA,DP05_0001PM,DP05_0001PMA,DP05_0002E,DP05_0002EA,...,DP05_0108PEA,DP05_0108PM,DP05_0108PMA,GEO_ID,NAME,state,public use microdata area,vintage,sample,group
0,159559,NaN,10759,NaN,159559,NaN,-888888888,NaN,77505,NaN,...,NaN,3.2,NaN,795P200US3604103,NYC-Manhattan Community District 3--Lower East...,36,4103,2024,acs1,group(DP05)
1,122452,NaN,11570,NaN,122452,NaN,-888888888,NaN,64015,NaN,...,NaN,4.3,NaN,795P200US3604104,NYC-Manhattan Community District 4--Chelsea & ...,36,4104,2024,acs1,group(DP05)
2,230436,NaN,14503,NaN,230436,NaN,-888888888,NaN,112428,NaN,...,NaN,2.1,NaN,795P200US3604107,NYC-Manhattan Community District 7--Upper West...,36,4107,2024,acs1,group(DP05)
3,224027,NaN,12229,NaN,224027,NaN,-888888888,NaN,99081,NaN,...,NaN,2.1,NaN,795P200US3604108,NYC-Manhattan Community District 8--Upper East...,36,4108,2024,acs1,group(DP05)
4,120387,NaN,12418,NaN,120387,NaN,-888888888,NaN,53605,NaN,...,NaN,4.5,NaN,795P200US3604109,NYC-Manhattan Community District 9--Morningsid...,36,4109,2024,acs1,group(DP05)


In [3]:
# labels
with open(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\HTM Projects\ACS_Warehouse\docs\dp_variables_2024.json", 'r') as f:
    data = json.load(f)

meta_df = (
    pd.DataFrame.from_dict(data["variables"], orient="index")
      .reset_index()
      .rename(columns={"index": "variable"})
)
meta_df

,variable,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,DP02_0126E,Estimate!!ANCESTRY!!Total population!!Arab,Selected Social Characteristics in the United ...,int,DP02,0,NaN,NaN,"DP02_0126EA,DP02_0126M,DP02_0126MA",NaN
4,DP05_0050PE,Percent!!RACE!!Total population!!One race!!Bla...,ACS Demographic and Housing Estimates,float,DP05,0,NaN,NaN,"DP05_0050PEA,DP05_0050PM,DP05_0050PMA",NaN
...,...,...,...,...,...,...,...,...,...,...
1412,DP03_0039PE,Percent!!INDUSTRY!!Civilian employed populatio...,Selected Economic Characteristics,float,DP03,0,NaN,NaN,"DP03_0039PEA,DP03_0039PM,DP03_0039PMA",NaN
1413,DP02_0098E,Estimate!!YEAR OF ENTRY!!Population born outsi...,Selected Social Characteristics in the United ...,int,DP02,0,NaN,NaN,"DP02_0098EA,DP02_0098M,DP02_0098MA",NaN
1414,DP04_0095PE,Percent!!SELECTED MONTHLY OWNER COSTS (SMOC)!!...,Selected Housing Characteristics,float,DP04,0,NaN,NaN,"DP04_0095PEA,DP04_0095PM,DP04_0095PMA",NaN
1415,DP02_0036PE,Percent!!MARITAL STATUS!!Females 15 years and ...,Selected Social Characteristics in the United ...,float,DP02,0,NaN,NaN,"DP02_0036PEA,DP02_0036PM,DP02_0036PMA",NaN


In [4]:
# Extract parameters for function and file name
data_sample = data_df['sample'][0]
data_vintage = data_df['vintage'][0]
data_geography = 'PUMA'
data_group = data_df['group'][0][6:-1]
# group_desc = 'SOCIAL_CHAR'                # DP02
# group_desc = 'ECONOMIC_CHAR'            # DP03
# group_desc = 'HOUSING_CHAR'             # DP04
group_desc = 'DEMOGRAPHIC_CHAR'         # DP05 

In [5]:
# Define Function to format tables
def acs_dp_to_wide(data_df:pd.DataFrame, meta_df:pd.DataFrame, sample:str, vintage:int) -> tuple:
    """
    Function is designed to work with data profile coming from census API requests
    
    Args:
        df_api: Data Profile DataFrame 
        df_meta: Metadata DataFrame
        sample: 1_yr or 5_yr
        vintage: survey year (or last year in 5_yr samples)
    
    Returns:
        pd.DataFrame: Wide format table (variables in rows, geographies in columns) with data profile data

    """
    # PREPARE DATA
    # Remove unnecessary columns from data df
    var_df = data_df.loc[:, ~data_df.columns.str.endswith(('EA', 'MA', 'PEA'))].copy()                   # drop annotation columns
    var_df = var_df.drop(columns = ['GEO_ID', 'state', 'vintage', 'NAME', 'sample', 'group'])            # drop other columns that would make transposing more difficult

    # Recode missing values
    var_df = var_df.replace(-888888888, np.nan)
    var_df = var_df.replace(-999999999, np.nan)
    
    # Make sure all variables are object (avoid issues with transpose)
    var_df = var_df.apply(pd.to_numeric, errors = 'coerce')
    
    # Transpose dataframe
    wide_df = var_df.transpose()
    headers = wide_df.iloc[-1].astype(int)                                          # type int to remove decimal point from column headers
    wide_df.columns = headers
    wide_df = wide_df[:-1].copy()                                                   # Remove row with headers
    wide_df = wide_df.add_prefix('puma_')
    wide_df = wide_df.reset_index().rename(columns={'index': 'variable'})           # reset index

    # Reincorporate vintage
    wide_df['vintage'] = vintage

    # Add sample indicator
    wide_df['sample'] = sample
    
    # Get variable base name
    wide_df['var_name_base'] = wide_df['variable'].str[0:9]

    # Name qualifier / variable type
    wide_df['var_type'] = wide_df['variable'].str.extract(r'\d+([A-Z]+)', expand=False)
    
    # Change description in var type
    wide_df['var_type'] = wide_df['var_type'].replace({
        'E': 'Estimate',
        'M': 'Estimate MOE',
        'PE': 'Percentage',
        'PM': 'Percentage MOE'
        })
    
    # PREPARE META DATA
    # Sort values
    meta_df = meta_df.sort_values(by = 'variable')
    
    # Keep only variables in groups DO02 to DP05
    meta_var_df = meta_df[meta_df['group'].isin(['DP02', 'DP03', 'DP04', 'DP05'])].copy()
    
    # Variable base name
    meta_var_df['var_name_base'] = meta_var_df['variable'].str[0:9]

    # Name qualifier / var type
    meta_var_df['var_type'] = meta_var_df['variable'].str.extract(r'\d+([A-Z]+)', expand=False)
    
    # Gen new label column without 'Estimate' or 'Percent' prefix
    meta_var_df['base_label'] = meta_var_df['label'].replace('Estimate|Percent', '', regex=True)
    
    # Keep only the columns we need
    meta_labels_df = meta_var_df[['var_name_base', 'base_label', 'group']].copy()
    
    # Drop duplicates
    meta_labels_df = meta_labels_df.drop_duplicates()
    
    # Label format: Remove initial '!!'
    meta_labels_df['base_label'] = meta_labels_df['base_label'].str.lstrip('!!')

    # Label format: Add a column for the topic of the variable (could be useful when applying filters in Excel)
    meta_labels_df['var_topic'] = meta_labels_df['base_label'].str.split('!!').str[0]

    # Label format: Replace '!!' with ' - '
    meta_labels_df['base_label'] = meta_labels_df['base_label'].str.replace('!!', ' - ', regex=True)

    
    
    # MERGE DATA WITH LABELS
    # Merge
    final_df = pd.merge(wide_df, meta_labels_df, how = 'left', left_on='var_name_base', right_on='var_name_base')
    
    # FORMAT FINAL DATA FRAME
    # Drop unnecessary columns
    final_df = final_df.drop(columns = ['var_name_base'])               # Used only for merge
    # rename columns
    final_df = final_df.rename(columns = {'base_label' : 'var_label', 'group' : 'var_group'})
    # sort columns
    final_df = final_df.sort_index(axis = 1, ascending = False)
    
    # GEO CODES
    geo_labels = data_df[['public use microdata area', 'state', 'NAME']]
    geo_labels = geo_labels.rename(columns={'public use microdata area' : 'puma_code', 'state' : 'state_code', 'NAME' : 'puma_name'} )
    
    return(final_df, geo_labels)  

In [6]:
dp_wide, geocodes = acs_dp_to_wide(data_df=data_df, meta_df=meta_df, sample=data_sample, vintage=data_vintage)

In [7]:
dp_wide

,vintage,variable,var_type,var_topic,var_label,var_group,sample,puma_4503,puma_4502,puma_4501,...,puma_4165,puma_4121,puma_4112,puma_4111,puma_4110,puma_4109,puma_4108,puma_4107,puma_4104,puma_4103
0,2024,DP05_0001E,Estimate,SEX AND AGE,SEX AND AGE - Total population,DP05,acs1,169613.0,142859.0,185740.0,...,202137.0,160189.0,182736.0,133493.0,125248.0,120387.0,224027.0,230436.0,122452.0,159559.0
1,2024,DP05_0001M,Estimate MOE,SEX AND AGE,SEX AND AGE - Total population,DP05,acs1,11769.0,9793.0,9651.0,...,13593.0,12225.0,12385.0,13077.0,12448.0,12418.0,12229.0,14503.0,11570.0,10759.0
2,2024,DP05_0001PE,Percentage,SEX AND AGE,SEX AND AGE - Total population,DP05,acs1,169613.0,142859.0,185740.0,...,202137.0,160189.0,182736.0,133493.0,125248.0,120387.0,224027.0,230436.0,122452.0,159559.0
3,2024,DP05_0001PM,Percentage MOE,SEX AND AGE,SEX AND AGE - Total population,DP05,acs1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024,DP05_0002E,Estimate,SEX AND AGE,SEX AND AGE - Total population - Male,DP05,acs1,83392.0,68340.0,91879.0,...,93411.0,79914.0,91883.0,67192.0,56638.0,53605.0,99081.0,112428.0,64015.0,77505.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,2024,DP05_0107PM,Percentage MOE,"CITIZEN, VOTING AGE POPULATION","CITIZEN, VOTING AGE POPULATION - Citizen, 18 a...",DP05,acs1,1.4,2.1,1.8,...,3.2,2.6,2.9,3.8,4.1,4.5,2.1,2.1,4.3,3.2
428,2024,DP05_0108E,Estimate,"CITIZEN, VOTING AGE POPULATION","CITIZEN, VOTING AGE POPULATION - Citizen, 18 a...",DP05,acs1,64296.0,52638.0,67974.0,...,88096.0,64076.0,61126.0,49383.0,49748.0,48629.0,99426.0,92974.0,44342.0,66011.0
429,2024,DP05_0108M,Estimate MOE,"CITIZEN, VOTING AGE POPULATION","CITIZEN, VOTING AGE POPULATION - Citizen, 18 a...",DP05,acs1,4399.0,4569.0,3841.0,...,8224.0,5209.0,4980.0,4959.0,6049.0,5714.0,7034.0,5964.0,5692.0,5091.0
430,2024,DP05_0108PE,Percentage,"CITIZEN, VOTING AGE POPULATION","CITIZEN, VOTING AGE POPULATION - Citizen, 18 a...",DP05,acs1,51.6,51.9,52.2,...,53.9,51.3,49.0,54.3,55.0,58.2,57.7,53.1,47.0,53.5


In [8]:
# Save data

directory = fr'C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\{data_sample}\{data_vintage}\{data_geography}'
file_name = f'{data_sample}_{data_vintage}_{data_geography}_NYC_{data_group}_{group_desc}.xlsx'

# Ensure the folder exists (create if missing)
os.makedirs(directory, exist_ok=True)


full_path = os.path.join(directory, file_name)

with pd.ExcelWriter(full_path) as writer:
    dp_wide.to_excel(writer, sheet_name = 'data', index = False)
    geocodes.to_excel(writer, sheet_name = 'geo_labels', index = False)